# 🫁 Pneumonia Detection - Model Training (EfficientNet + SVM)

Welcome to your Final Year Project training notebook! This notebook will:
1. Setup the dataset
2. Extract features using **EfficientNetB0**
3. Train a **Linear SVM** classifier
4. Save the model for your Streamlit Dashboard.

## Step 1: Install Requirements
Google Colab has most libraries, but we might need a few extras.

In [ ]:
!pip install fpdf joblib opencv-python-headless

## Step 2: Import Libraries

In [ ]:
import os
import numpy as np
import joblib
import cv2
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.preprocessing.image import img_to_array, load_img
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

## Step 3: Define Helper Functions
(These are simplified versions of your `src/` modules for Colab compatibility)

In [ ]:
def load_and_preprocess_image(image_path, target_size=(224, 224)):
    img = load_img(image_path, target_size=target_size)
    img_array = img_to_array(img)
    img_array = img_array / 255.0  # Normalize
    return np.expand_dims(img_array, axis=0)

def get_data_labels(directory):
    filepaths = []
    labels = []
    for label_name in ['NORMAL', 'PNEUMONIA']:
        label_val = 0 if label_name == 'NORMAL' else 1
        class_dir = os.path.join(directory, label_name)
        if not os.path.exists(class_dir): continue
        for img_name in os.listdir(class_dir):
            if img_name.lower().endswith(('.jpeg', '.jpg', '.png')):
                filepaths.append(os.path.join(class_dir, img_name))
                labels.append(label_val)
    return filepaths, np.array(labels)

## Step 4: Load Dataset Paths
**Note:** Please upload your `chest_xray` folder to the left sidebar in Colab before running this.

In [ ]:
# Update these paths if you uploaded files differently
TRAIN_DIR = "chest_xray/train"
TEST_DIR = "chest_xray/test"

print("Indexing images...")
train_files, train_labels = get_data_labels(TRAIN_DIR)
test_files, test_labels = get_data_labels(TEST_DIR)

print(f"Found {len(train_files)} training images.")
print(f"Found {len(test_files)} testing images.")

## Step 5: Feature Extraction using EfficientNetB0

In [ ]:
print("Loading EfficientNetB0 (Transfer Learning)...")
model = EfficientNetB0(weights='imagenet', include_top=False, pooling='avg', input_shape=(224, 224, 3))

def extract_features_batch(files):
    features = []
    total = len(files)
    for i, f in enumerate(files):
        img = load_and_preprocess_image(f)
        feat = model.predict(img, verbose=0)
        features.append(feat.flatten())
        if i % 100 == 0: print(f"Progress: {i}/{total}")
    return np.array(features)

print("Extracting features from Training Set (This will take a few minutes)... ")
X_train = extract_features_batch(train_files)

print("Extracting features from Testing Set...")
X_test = extract_features_batch(test_files)

## Step 6: Train SVM Classifier

In [ ]:
print("Training Linear SVM...")
svm = SVC(kernel='linear', probability=True)
svm.fit(X_train, train_labels)

print("Training complete!")

## Step 7: Evaluate Results

In [ ]:
y_pred = svm.predict(X_test)
print("\n--- Accuracy Score ---")
print(f"{accuracy_score(test_labels, y_pred) * 100:.2f}%")

print("\n--- Classification Report ---")
print(classification_report(test_labels, y_pred, target_names=['NORMAL', 'PNEUMONIA']))

# Plot Confusion Matrix
cm = confusion_matrix(test_labels, y_pred)
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Normal', 'Pneumonia'], yticklabels=['Normal', 'Pneumonia'])
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

## Step 8: Save Model
Download this `.pkl` file and put it in your project folder on your PC to run the Streamlit app.

In [ ]:
joblib.dump(svm, "pneumonia_svm_model.pkl")
print("Model saved as 'pneumonia_svm_model.pkl'. Please download it from the sidebar!")